In [1]:
import subprocess
import librosa
import numpy as np
import json
from pytube import YouTube
import os
import time
import re
import urllib.parse as urlparse

# Sample JSON timestamps for the first recording
timestamps_json = '''
[{"t":0,"mix":0},{"t":6.163,"mix":1},{"t":10.761,"mix":2},{"t":15.238,"mix":3},
{"t":19.925,"mix":4},{"t":24.313,"mix":5},{"t":28.731,"mix":6},{"t":33.072,"mix":7},{"t":40.075,
"mix":8},{"t":44.674,"mix":9},{"t":49.045,"mix":10},{"t":53.78,"mix":11},{"t":58.842,"mix":12},
{"t":63.39,"mix":13},{"t":67.851,"mix":14},{"t":73.048,"mix":15},{"t":77.722,"mix":16},{"t":82.581,
"mix":17},{"t":87.302,"mix":18},{"t":91.786,"mix":19},{"t":96.662,"mix":20},{"t":101.12,"mix":21},
{"t":105.536,"mix":22},{"t":109.847,"mix":23},{"t":114.254,"mix":24},{"t":118.838,"mix":25},
{"t":123.009,"mix":26},{"t":127.322,"mix":27},{"t":132.247,"mix":28},{"t":136.66,"mix":29},
{"t":141.17,"mix":30},{"t":145.66,"mix":31},{"t":150.209,"mix":32},{"t":154.864,"mix":33},
{"t":159.149,"mix":34},{"t":163.867,"mix":35},{"t":168.327,"mix":36},{"t":172.873,"mix":37},
{"t":174.306,"mix":38},{"t":175.46,"mix":39},{"t":176.713,"mix":40},{"t":177.96,"mix":41},
{"t":179.123,"mix":42},{"t":180.256,"mix":43},{"t":181.545,"mix":44},{"t":182.577,"mix":45},
{"t":183.687,"mix":46},{"t":184.756,"mix":47},{"t":185.94,"mix":48},{"t":187.124,"mix":49},
{"t":188.201,"mix":50},{"t":189.311,"mix":51},{"t":190.517,"mix":52},{"t":191.621,"mix":53},
{"t":192.878,"mix":54},{"t":193.955,"mix":55},{"t":195.089,"mix":56},{"t":196.312,"mix":57},
{"t":197.516,"mix":58},{"t":198.756,"mix":59},{"t":199.957,"mix":60},{"t":201.162,"mix":61},
{"t":202.306,"mix":62},{"t":203.508,"mix":63},{"t":204.72,"mix":64},{"t":205.883,"mix":65},
{"t":206.991,"mix":66},{"t":208.069,"mix":67},{"t":209.141,"mix":68},{"t":210.244,"mix":69},
{"t":211.552,"mix":70},{"t":212.856,"mix":71},{"t":214.036,"mix":72},{"t":215.051,"mix":73},
{"t":216.072,"mix":74},{"t":217.181,"mix":75},{"t":218.297,"mix":76},{"t":219.462,"mix":77},
{"t":220.644,"mix":78},{"t":221.771,"mix":79},{"t":222.898,"mix":80},{"t":224.064,"mix":81},
{"t":225.238,"mix":82},{"t":226.39,"mix":83},{"t":227.591,"mix":84},{"t":228.751,"mix":85},
{"t":229.88,"mix":86},{"t":231.067,"mix":87},{"t":232.276,"mix":88},{"t":233.456,"mix":89},
{"t":234.673,"mix":90},{"t":235.801,"mix":91},{"t":236.991,"mix":92},{"t":238.189,"mix":93},
{"t":239.345,"mix":94},{"t":240.435,"mix":95},{"t":241.638,"mix":96},{"t":242.804,"mix":97},
{"t":243.963,"mix":98},{"t":245.17,"mix":99},{"t":246.29,"mix":100},{"t":247.483,"mix":101},
{"t":248.768,"mix":102},{"t":250.257,"mix":103},{"t":251.517,"mix":104},{"t":252.659,"mix":105},
{"t":253.889,"mix":106},{"t":255.131,"mix":107},{"t":256.286,"mix":108},{"t":257.606,"mix":109},
{"t":258.916,"mix":110},{"t":260.193,"mix":111},{"t":261.629,"mix":112},{"t":262.873,"mix":113},
{"t":264.063,"mix":114},{"t":265.291,"mix":115},{"t":266.468,"mix":116},{"t":267.72,"mix":117},
{"t":269.022,"mix":118},{"t":270.308,"mix":119},{"t":271.652,"mix":120},{"t":273.215,"mix":121},
{"t":274.724,"mix":122},{"t":276.241,"mix":123},{"t":277.884,"mix":124},{"t":279.304,"mix":125},
{"t":280.818,"mix":126},{"t":282.667,"mix":127},{"t":284.391,"mix":128},{"t":285.706,"mix":129},
{"t":287.015,"mix":130},{"t":288.271,"mix":131},{"t":289.542,"mix":132},{"t":290.844,"mix":133},
{"t":292.235,"mix":134},{"t":293.627,"mix":135},{"t":294.935,"mix":136},{"t":296.237,"mix":137},
{"t":297.612,"mix":138},{"t":298.939,"mix":139},{"t":300.287,"mix":140},{"t":301.67,"mix":141},
{"t":303.196,"mix":142},{"t":304.622,"mix":143},{"t":306.01,"mix":144},{"t":307.418,"mix":145},
{"t":309.005,"mix":146},{"t":310.57,"mix":147},{"t":312.122,"mix":148},{"t":313.772,"mix":149},
{"t":315.484,"mix":150},{"t":317.174,"mix":151},{"t":318.807,"mix":152},{"t":320.415,"mix":153},
{"t":323.094,"mix":154},{"t":325.048,"mix":155},{"t":326.796,"mix":156},{"t":327.897,"mix":157},
{"t":329.15,"mix":158},{"t":330.299,"mix":159},{"t":331.341,"mix":160},{"t":332.427,"mix":161},
{"t":333.536,"mix":162},{"t":334.641,"mix":163},{"t":335.651,"mix":164},{"t":336.782,"mix":165},
{"t":337.727,"mix":166},{"t":338.804,"mix":167},{"t":339.84,"mix":168},{"t":340.87,"mix":169},
{"t":341.99,"mix":170},{"t":343.074,"mix":171},{"t":344.082,"mix":172},{"t":345.211,"mix":173},
{"t":346.222,"mix":174},{"t":347.306,"mix":175},{"t":348.331,"mix":176},{"t":349.347,"mix":177},
{"t":350.429,"mix":178},{"t":351.464,"mix":179},{"t":352.571,"mix":180},{"t":353.58,"mix":181},
{"t":354.648,"mix":182},{"t":355.734,"mix":183},{"t":356.814,"mix":184},{"t":357.881,"mix":185},
{"t":359.003,"mix":186},{"t":360.189,"mix":187},{"t":361.305,"mix":190},{"t":362.552,"mix":191},
{"t":363.8,"mix":192},{"t":365.076,"mix":193},{"t":366.173,"mix":194},{"t":367.339,"mix":195},
{"t":368.461,"mix":196},{"t":369.726,"mix":197},{"t":370.8,"mix":198},{"t":372.159,"mix":199},
{"t":373.366,"mix":200},{"t":374.658,"mix":201},{"t":375.87,"mix":202},{"t":377.08,"mix":203},
{"t":378.172,"mix":204},{"t":379.505,"mix":205},{"t":380.682,"mix":206},{"t":381.931,"mix":207},
{"t":383.162,"mix":208},{"t":384.506,"mix":209},{"t":385.67,"mix":210},{"t":386.839,"mix":211},
{"t":388.186,"mix":212},{"t":389.412,"mix":213},{"t":390.596,"mix":214},{"t":391.794,"mix":215},
{"t":392.948,"mix":216},{"t":394.245,"mix":217},{"t":395.44,"mix":218},{"t":396.711,"mix":219},
{"t":397.959,"mix":220},{"t":399.179,"mix":221},{"t":400.443,"mix":222},{"t":401.666,"mix":223},
{"t":402.862,"mix":224},{"t":404.135,"mix":225},{"t":405.251,"mix":226},{"t":406.509,"mix":227},
{"t":407.613,"mix":228},{"t":408.751,"mix":229},{"t":409.915,"mix":230},{"t":411.037,"mix":231},
{"t":412.133,"mix":232},{"t":413.315,"mix":233},{"t":414.476,"mix":234},{"t":415.624,"mix":235},
{"t":416.686,"mix":236},{"t":417.768,"mix":237},{"t":418.857,"mix":238},{"t":419.965,"mix":239},
{"t":421.135,"mix":240},{"t":422.214,"mix":241},{"t":423.29,"mix":242},{"t":424.41,"mix":243},
{"t":425.492,"mix":244},{"t":426.619,"mix":245},{"t":427.709,"mix":246},{"t":428.763,"mix":247},
{"t":429.89,"mix":248},{"t":430.99,"mix":249},{"t":432.055,"mix":250},{"t":433.203,"mix":251},
{"t":434.295,"mix":252},{"t":435.495,"mix":253},{"t":436.6,"mix":254},{"t":437.705,"mix":255},
{"t":438.855,"mix":256},{"t":439.98,"mix":257},{"t":441.059,"mix":258},{"t":442.174,"mix":259},
{"t":443.309,"mix":260},{"t":444.418,"mix":261},{"t":445.552,"mix":262},{"t":446.715,"mix":263},
{"t":447.891,"mix":264},{"t":449.048,"mix":265},{"t":450.099,"mix":266},{"t":451.194,"mix":267},
{"t":452.293,"mix":268},{"t":453.461,"mix":269},{"t":454.552,"mix":270},{"t":455.645,"mix":271},
{"t":456.867,"mix":272},{"t":458.162,"mix":273},{"t":459.394,"mix":274},{"t":460.681,"mix":275},
{"t":461.883,"mix":276},{"t":463.058,"mix":277},{"t":464.384,"mix":278},{"t":465.674,"mix":279},
{"t":466.964,"mix":280},{"t":468.181,"mix":281},{"t":469.335,"mix":282},{"t":470.611,"mix":283},
{"t":471.896,"mix":284},{"t":473.176,"mix":285},{"t":474.402,"mix":286},{"t":475.628,"mix":287},
{"t":476.901,"mix":288},{"t":478.245,"mix":289},{"t":479.37,"mix":290},{"t":480.623,"mix":291},
{"t":482.113,"mix":292},{"t":483.554,"mix":293},{"t":484.909,"mix":294},{"t":486.468,"mix":295},
{"t":487.904,"mix":296},{"t":489.273,"mix":297},{"t":490.379,"mix":298},{"t":491.845,"mix":299},
{"t":492.897,"mix":300},{"t":494.35,"mix":301},{"t":495.586,"mix":302},{"t":496.849,"mix":303},
{"t":497.991,"mix":304},{"t":499.178,"mix":305},{"t":500.211,"mix":306},{"t":501.405,"mix":307},
{"t":502.542,"mix":308},{"t":503.729,"mix":309},{"t":504.781,"mix":310},{"t":505.939,"mix":311},
{"t":507.07,"mix":312},{"t":508.23,"mix":313},{"t":509.315,"mix":314},{"t":510.495,"mix":315},
{"t":511.581,"mix":316},{"t":512.809,"mix":317},{"t":513.919,"mix":318},{"t":515.058,"mix":319},
{"t":516.144,"mix":320},{"t":517.324,"mix":321},{"t":518.505,"mix":322},{"t":519.643,"mix":323},
{"t":520.86,"mix":324},{"t":521.94,"mix":325},{"t":523.133,"mix":326},{"t":524.215,"mix":327},
{"t":525.299,"mix":328},{"t":526.472,"mix":329},{"t":527.623,"mix":330},{"t":528.787,"mix":331},
{"t":529.9,"mix":332},{"t":531.012,"mix":333},{"t":532.111,"mix":334},{"t":533.333,"mix":335},
{"t":534.55,"mix":336},{"t":535.778,"mix":337},{"t":537.005,"mix":338},{"t":538.209,"mix":339},
{"t":539.323,"mix":340},{"t":540.481,"mix":341},{"t":541.573,"mix":342},{"t":542.808,"mix":343},
{"t":543.973,"mix":344},{"t":545.179,"mix":345},{"t":546.311,"mix":346},{"t":547.403,"mix":347},
{"t":548.447,"mix":348},{"t":549.491,"mix":349},{"t":550.622,"mix":350},{"t":551.716,"mix":351},
{"t":552.843,"mix":352},{"t":553.963,"mix":353},{"t":555.07,"mix":354},{"t":556.208,"mix":355},
{"t":557.314,"mix":356},{"t":558.433,"mix":357},{"t":559.557,"mix":358},{"t":560.729,"mix":359},
{"t":561.871,"mix":360},{"t":563.072,"mix":361},{"t":564.194,"mix":362},{"t":565.407,"mix":363},
{"t":566.61,"mix":364},{"t":567.773,"mix":365},{"t":568.926,"mix":366},{"t":570.052,"mix":367},
{"t":571.162,"mix":368},{"t":572.274,"mix":369},{"t":573.363,"mix":370},{"t":574.535,"mix":371},
{"t":575.636,"mix":372},{"t":576.77,"mix":373},{"t":577.916,"mix":374},{"t":579.068,"mix":375},
{"t":580.193,"mix":376},{"t":581.414,"mix":377},{"t":582.607,"mix":378},{"t":583.976,"mix":379},
{"t":585.184,"mix":380},{"t":586.327,"mix":381},{"t":587.5,"mix":382},{"t":588.638,"mix":383},
{"t":589.902,"mix":384},{"t":591.125,"mix":385},{"t":592.46,"mix":386},{"t":593.833,"mix":387},
{"t":595.108,"mix":388},{"t":596.253,"mix":389},{"t":597.386,"mix":390},{"t":598.58,"mix":391},
{"t":599.832,"mix":392},{"t":601.071,"mix":393},{"t":602.32,"mix":394},{"t":603.773,"mix":395},
{"t":605.228,"mix":396},{"t":606.726,"mix":397},{"t":608.108,"mix":398},{"t":609.413,"mix":399},
{"t":610.865,"mix":400},{"t":612.406,"mix":401},{"t":614.243,"mix":402},{"t":616.03,"mix":403},
{"t":617.479,"mix":404},{"t":618.6,"mix":405},{"t":619.956,"mix":406},{"t":621.285,"mix":407},
{"t":622.561,"mix":408},{"t":624.01,"mix":409},{"t":625.353,"mix":410},{"t":626.689,"mix":411},
{"t":627.938,"mix":412},{"t":629.293,"mix":413},{"t":630.628,"mix":414},{"t":631.966,"mix":415},
{"t":633.305,"mix":416},{"t":634.953,"mix":417},{"t":636.34,"mix":418},{"t":637.746,"mix":419},
{"t":639.292,"mix":420},{"t":640.797,"mix":421},{"t":642.284,"mix":422},{"t":644.121,"mix":423},
{"t":645.8,"mix":424},{"t":647.166,"mix":425},{"t":648.945,"mix":426},{"t":650.533,"mix":427},
{"t":652.209,"mix":428},{"t":654.036,"mix":429},{"t":656.137,"mix":430},{"t":657.985,"mix":431},
{"t":659.239,"mix":432},{"t":660.338,"mix":433},{"t":661.506,"mix":434},{"t":662.623,"mix":435},
{"t":663.724,"mix":436},{"t":664.895,"mix":437},{"t":665.948,"mix":438},{"t":666.92,"mix":439},
{"t":668.065,"mix":440},{"t":669.087,"mix":441},{"t":670.175,"mix":442},{"t":671.206,"mix":443},
{"t":672.173,"mix":444},{"t":673.282,"mix":445},{"t":674.397,"mix":446},{"t":675.434,"mix":447},
{"t":676.568,"mix":448},{"t":677.619,"mix":449},{"t":678.613,"mix":450},{"t":679.644,"mix":451},
{"t":680.65,"mix":452},{"t":681.757,"mix":453},{"t":682.786,"mix":454},{"t":683.799,"mix":455},
{"t":684.827,"mix":456},{"t":685.848,"mix":457},{"t":686.91,"mix":458},{"t":688.029,"mix":459},
{"t":689.142,"mix":460},{"t":690.307,"mix":461},{"t":691.364,"mix":462},{"t":692.559,"mix":463},
{"t":693.779,"mix":464},{"t":694.919,"mix":465},{"t":696.078,"mix":466},{"t":697.227,"mix":467},
{"t":698.355,"mix":468},{"t":699.466,"mix":469},{"t":700.568,"mix":470},{"t":701.644,"mix":471},
{"t":702.712,"mix":472},{"t":703.827,"mix":473},{"t":704.923,"mix":474},{"t":706.109,"mix":475},
{"t":707.527,"mix":476},{"t":708.872,"mix":477},{"t":710.21,"mix":478},{"t":711.497,"mix":479},
{"t":713.024,"mix":480},{"t":714.414,"mix":481},{"t":715.763,"mix":482},{"t":717.099,"mix":483},
{"t":718.492,"mix":484},{"t":719.762,"mix":485},{"t":721.11,"mix":486},{"t":722.643,"mix":487},
{"t":723.903,"mix":488},{"t":725.33,"mix":489},{"t":726.758,"mix":490},{"t":728.171,"mix":491},
{"t":729.672,"mix":492},{"t":731.266,"mix":493},{"t":733.06,"mix":494},{"t":734.869,"mix":495},
{"t":737.885,"mix":496},{"t":740.55,"mix":497},{"t":743.043,"mix":498},{"t":745.699,"mix":499},
{"t":748.212,"mix":500},{"t":751.017,"mix":501},{"t":753.59,"mix":502},{"t":756.359,"mix":503},
{"t":758.954,"mix":504},{"t":761.61,"mix":505},{"t":764.117,"mix":506},{"t":766.881,"mix":507},
{"t":769.579,"mix":508},{"t":772.637,"mix":509},{"t":775.878,"mix":510},{"t":779.257,"mix":511},
{"t":783.081,"mix":512},{"t":794.793,"mix":513},{"t":800.236,"mix":514},{"t":804.506,"mix":515},
{"t":808.382,"mix":516},{"t":812.278,"mix":517},{"t":815.582,"mix":518},{"t":819.753,"mix":519},
{"t":823.441,"mix":520},{"t":827.047,"mix":521},{"t":830.593,"mix":522},{"t":834.1,"mix":523},
{"t":837.711,"mix":524},{"t":841.595,"mix":525},{"t":845.323,"mix":526},{"t":849.056,"mix":527},
{"t":853.21,"mix":528},{"t":857.335,"mix":529},{"t":861.441,"mix":530},{"t":865.116,"mix":531},
{"t":869.034,"mix":532},{"t":872.738,"mix":533},{"t":876.698,"mix":534},{"t":880.247,"mix":535},
{"t":884.553,"mix":536},{"t":888.705,"mix":537},{"t":892.948,"mix":538},{"t":896.887,"mix":539},
{"t":901.078,"mix":540},{"t":904.648,"mix":541},{"t":908.215,"mix":542},{"t":911.734,"mix":543},
{"t":915.37,"mix":544},{"t":918.734,"mix":545},{"t":922.124,"mix":546},{"t":925.458,"mix":547},
{"t":928.769,"mix":548},{"t":932.066,"mix":549},{"t":936.287,"mix":550},{"t":941.07,"mix":551},
{"t":944.739,"mix":552},{"t":948.589,"mix":553},{"t":952.454,"mix":554},{"t":956.204,"mix":555},
{"t":960.007,"mix":556},{"t":963.701,"mix":557},{"t":967.286,"mix":558},{"t":970.88,"mix":559},
{"t":974.784,"mix":560},{"t":978.837,"mix":561},{"t":982.344,"mix":562},{"t":986.315,"mix":563},
{"t":990.102,"mix":564},{"t":993.854,"mix":565},{"t":997.581,"mix":566},{"t":1000.882,"mix":567},
{"t":1004.521,"mix":568},{"t":1008.39,"mix":569},{"t":1011.992,"mix":570},{"t":1015.512,"mix":571},
{"t":1018.961,"mix":572},{"t":1022.777,"mix":573},{"t":1026.988,"mix":574},{"t":1031.914,
"mix":575},{"t":1036.523,"mix":576},{"t":1041.203,"mix":577},{"t":1046.321,"mix":578},{"t":1051.887,
"mix":579},{"t":1056.056,"mix":580},{"t":1060.333,"mix":581},{"t":1064,"mix":582},{"t":1068.256,
"mix":583},{"t":1072.282,"mix":584},{"t":1075.648,"mix":585},{"t":1079.693,"mix":586},{"t":1083.31,
"mix":587},{"t":1086.645,"mix":588},{"t":1090.438,"mix":589},{"t":1094.261,"mix":590},{"t":1098.051,
"mix":591},{"t":1101.733,"mix":592},{"t":1105.484,"mix":593},{"t":1109.196,"mix":594},{"t":1113.232,
"mix":595},{"t":1117.079,"mix":596},{"t":1121.111,"mix":597},{"t":1124.302,"mix":598},{"t":1127.537,
"mix":599},{"t":1131.112,"mix":600},{"t":1134.878,"mix":601},{"t":1138.735,"mix":602},{"t":1144.019,
"mix":603},{"t":1148.109,"mix":604},{"t":1152.049,"mix":605},{"t":1156.049,"mix":606},{"t":1160.123,
"mix":607},{"t":1164.034,"mix":608},{"t":1168.531,"mix":609},{"t":1172.76,"mix":610},{"t":1177.187,
"mix":611},{"t":1181.344,"mix":612},{"t":1185.241,"mix":613},{"t":1189.517,"mix":614},{"t":1193.59,
"mix":615},{"t":1197.478,"mix":616},{"t":1201.727,"mix":617},{"t":1205.455,"mix":618},{"t":1209.243,
"mix":619},{"t":1212.96,"mix":620},{"t":1216.937,"mix":621},{"t":1220.525,"mix":622},{"t":1224.043,
"mix":623},{"t":1228.897,"mix":624},{"t":1233.227,"mix":625},{"t":1237.408,"mix":626},{"t":1241.706,
"mix":627},{"t":1245.6,"mix":628},{"t":1249.206,"mix":629},{"t":1253.218,"mix":630},{"t":1257.494,
"mix":631},{"t":1261.473,"mix":632},{"t":1265.431,"mix":633},{"t":1269.42,"mix":634},{"t":1275.141,
"mix":635},{"t":1281.481,"mix":636},{"t":1287.304,"mix":637},{"t":1293.128,"mix":638},{"t":1298.338,
"mix":639},{"t":1305.258,"mix":640},{"t":1325.939,"mix":641},{"t":1329.229,"mix":642},{"t":1330.467,
"mix":643},{"t":1332.047,"mix":644},{"t":1333.575,"mix":645},{"t":1334.968,"mix":646},{"t":1336.385,
"mix":647},{"t":1337.739,"mix":648},{"t":1339.139,"mix":649},{"t":1340.565,"mix":650},{"t":1341.98,
"mix":651},{"t":1343.419,"mix":652},{"t":1344.913,"mix":653},{"t":1346.3,"mix":654},{"t":1347.778,
"mix":655},{"t":1349.147,"mix":656},{"t":1350.639,"mix":657},{"t":1352.003,"mix":658},{"t":1353.357,
"mix":659},{"t":1354.896,"mix":660},{"t":1356.26,"mix":661},{"t":1357.627,"mix":662},{"t":1359.053,
"mix":663},{"t":1360.574,"mix":664},{"t":1361.942,"mix":665},{"t":1363.337,"mix":666},{"t":1364.7,
"mix":667},{"t":1366.097,"mix":668},{"t":1367.492,"mix":669},{"t":1368.948,"mix":670},{"t":1370.335,
"mix":671},{"t":1371.644,"mix":672},{"t":1372.977,"mix":673},{"t":1374.406,"mix":674},{"t":1375.828,
"mix":675},{"t":1377.269,"mix":676},{"t":1378.636,"mix":677},{"t":1379.952,"mix":678},{"t":1381.36,
"mix":679},{"t":1382.85,"mix":680},{"t":1384.055,"mix":681},{"t":1385.511,"mix":682},{"t":1386.87,
"mix":683},{"t":1388.238,"mix":684},{"t":1389.638,"mix":685},{"t":1391.015,"mix":686},{"t":1392.419,
"mix":687},{"t":1393.839,"mix":688},{"t":1395.383,"mix":689},{"t":1396.68,"mix":690},{"t":1398.211,
"mix":691},{"t":1399.662,"mix":692},{"t":1401.058,"mix":693},{"t":1402.488,"mix":694},{"t":1403.915,
"mix":695},{"t":1405.365,"mix":696},{"t":1406.819,"mix":697},{"t":1408.326,"mix":698},{"t":1409.802,
"mix":699},{"t":1411.191,"mix":700},{"t":1412.749,"mix":701},{"t":1414.311,"mix":702},{"t":1415.858,
"mix":703},{"t":1417.413,"mix":704},{"t":1418.801,"mix":705},{"t":1420.439,"mix":706},{"t":1421.938,
"mix":707},{"t":1423.392,"mix":708},{"t":1424.849,"mix":709},{"t":1426.355,"mix":710},{"t":1427.948,
"mix":711},{"t":1429.502,"mix":712},{"t":1430.956,"mix":713},{"t":1432.331,"mix":714},{"t":1433.719,
"mix":715},{"t":1435.211,"mix":716},{"t":1436.698,"mix":717},{"t":1438.072,"mix":718},{"t":1439.401,
"mix":719},{"t":1440.97,"mix":720},{"t":1442.427,"mix":721},{"t":1443.755,"mix":722},{"t":1444.99,
"mix":723},{"t":1446.456,"mix":724},{"t":1447.805,"mix":725},{"t":1449.087,"mix":726},{"t":1450.323,
"mix":727},{"t":1451.795,"mix":728},{"t":1453.237,"mix":729},{"t":1454.594,"mix":730},{"t":1455.998,
"mix":731},{"t":1457.443,"mix":732},{"t":1458.801,"mix":733},{"t":1460.15,"mix":734},{"t":1461.453,
"mix":735},{"t":1462.887,"mix":736},{"t":1464.265,"mix":737},{"t":1465.65,"mix":738},{"t":1466.998,
"mix":739},{"t":1468.269,"mix":740},{"t":1469.645,"mix":741},{"t":1470.947,"mix":742},{"t":1472.336,
"mix":743},{"t":1473.77,"mix":744},{"t":1475.095,"mix":745},{"t":1476.343,"mix":746},{"t":1477.774,
"mix":747},{"t":1479.061,"mix":748},{"t":1480.493,"mix":727},{"t":1481.871,"mix":728},{"t":1483.144,
"mix":729},{"t":1484.386,"mix":730},{"t":1485.708,"mix":731},{"t":1487.028,"mix":732},{"t":1488.358,
"mix":733},{"t":1489.629,"mix":734},{"t":1490.919,"mix":735},{"t":1492.251,"mix":736},{"t":1493.568,
"mix":737},{"t":1494.809,"mix":738},{"t":1496.048,"mix":739},{"t":1497.323,"mix":740},{"t":1498.566,
"mix":741},{"t":1499.798,"mix":742},{"t":1501.131,"mix":743},{"t":1502.499,"mix":744},{"t":1503.801,
"mix":745},{"t":1505.089,"mix":746},{"t":1506.485,"mix":747},{"t":1507.904,"mix":749},{"t":1509.956,
"mix":750},{"t":1511.871,"mix":751},{"t":1513.611,"mix":752},{"t":1515.118,"mix":753},{"t":1516.62,
"mix":754},{"t":1518.177,"mix":755},{"t":1519.789,"mix":756},{"t":1521.383,"mix":757},{"t":1522.785,
"mix":758},{"t":1524.136,"mix":759},{"t":1525.664,"mix":760},{"t":1527.074,"mix":761},{"t":1528.533,
"mix":762},{"t":1529.985,"mix":763},{"t":1531.417,"mix":764},{"t":1532.881,"mix":765},{"t":1534.418,
"mix":766},{"t":1535.878,"mix":767},{"t":1537.318,"mix":768},{"t":1538.863,"mix":769},{"t":1540.383,
"mix":770},{"t":1541.761,"mix":771},{"t":1543.192,"mix":772},{"t":1544.621,"mix":773},{"t":1546.133,
"mix":774},{"t":1547.607,"mix":775},{"t":1548.98,"mix":776},{"t":1550.434,"mix":777},{"t":1551.886,
"mix":778},{"t":1553.302,"mix":779},{"t":1554.823,"mix":780},{"t":1556.364,"mix":781},{"t":1557.93,
"mix":782},{"t":1559.423,"mix":783},{"t":1561.046,"mix":784},{"t":1563.24,"mix":785},{"t":1564.999,
"mix":786},{"t":1566.599,"mix":787},{"t":1568.165,"mix":788},{"t":1569.809,"mix":789},{"t":1571.441,
"mix":790},{"t":1573.067,"mix":791},{"t":1574.773,"mix":792},{"t":1576.514,"mix":793},{"t":1578.318,
"mix":794},{"t":1580.275,"mix":795},{"t":1583.039,"mix":796},{"t":1584.999,"mix":797},{"t":1586.912,
"mix":798},{"t":1588.915,"mix":799},{"t":1591.062,"mix":800},{"t":1593.029,"mix":801},{"t":1595.015,
"mix":802},{"t":1597.159,"mix":803},{"t":1599.447,"mix":804},{"t":1602.459,"mix":805},{"t":1612.229,
"mix":806},{"t":1622.49,"mix":807},{"t":1630.612,"mix":808},{"t":1638.044,"mix":809},{"t":1645.746,
"mix":810},{"t":1653.921,"mix":811},{"t":1660.902,"mix":812},{"t":1667.703,"mix":813},{"t":1671.664,
"mix":814},{"t":1673.757,"mix":815},{"t":1675.593,"mix":816},{"t":1677.351,"mix":817},{"t":1685.458,
"mix":818},{"t":1693.6,"mix":819},{"t":1701.248,"mix":820},{"t":1709.821,"mix":821},{"t":1716.656,
"mix":822},{"t":1722.651,"mix":823},{"t":1725.756,"mix":824},{"t":1728.413,"mix":825},{"t":1733.2,
"mix":826},{"t":1738.173,"mix":827},{"t":1742.816,"mix":828},{"t":1746.944,"mix":829},{"t":1750.922,
"mix":830},{"t":1754.823,"mix":831},{"t":1758.497,"mix":832},{"t":1762.776,"mix":833},{"t":1767.725,
"mix":834},{"t":1775.878,"mix":835},{"t":1780.775,"mix":836},{"t":1785.27,"mix":837},{"t":1789.637,
"mix":838},{"t":1794.963,"mix":839},{"t":1800.314,"mix":840},{"t":1805.307,"mix":841},{"t":1809.481,
"mix":842},{"t":1814.656,"mix":843},{"t":1819.885,"mix":844},{"t":1823.965,"mix":845},{"t":1828.484,
"mix":846},{"t":1833.492,"mix":847},{"t":1838.153,"mix":848},{"t":1842.986,"mix":849},{"t":1847.967,
"mix":850},{"t":1852.826,"mix":851},{"t":1857.957,"mix":852},{"t":1863.312,"mix":853},{"t":1867.44,
"mix":854},{"t":1872.411,"mix":855},{"t":1877.126,"mix":856},{"t":1880.946,"mix":857},{"t":1885.265,
"mix":858},{"t":1889.126,"mix":859},{"t":1892.849,"mix":860},{"t":1896.526,"mix":861},{"t":1900.729,
"mix":862},{"t":1904.931,"mix":863},{"t":1909.17,"mix":864},{"t":1913.629,"mix":865},{"t":1919.191,
"mix":866},{"t":1922.319,"mix":867},{"t":1923.705,"mix":868},{"t":1926.265,"mix":869},{"t":1928.552,
"mix":870},{"t":1930.753,"mix":871},{"t":1933.113,"mix":872},{"t":1935.241,"mix":873},{"t":1937.633,
"mix":874},{"t":1939.569,"mix":875},{"t":1941.932,"mix":876},{"t":1944.031,"mix":877},{"t":1946.105,
"mix":878},{"t":1948.225,"mix":879},{"t":1950.3,"mix":880},{"t":1952.397,"mix":881},{"t":1954.622,
"mix":882},{"t":1956.7,"mix":883},{"t":1958.742,"mix":884},{"t":1960.715,"mix":885},{"t":1962.81,
"mix":886},{"t":1964.714,"mix":887},{"t":1966.812,"mix":888},{"t":1968.775,"mix":889},{"t":1970.78,
"mix":890},{"t":1972.666,"mix":891},{"t":1974.669,"mix":892},{"t":1976.594,"mix":893},{"t":1978.367,
"mix":894},{"t":1980.24,"mix":895},{"t":1982.051,"mix":896},{"t":1983.671,"mix":897},{"t":1985.378,
"mix":898},{"t":1986.892,"mix":899},{"t":1988.558,"mix":900},{"t":1990.314,"mix":901},{"t":1991.975,
"mix":902},{"t":1993.461,"mix":903},{"t":1995.149,"mix":904},{"t":1996.859,"mix":905},{"t":1998.486,
"mix":906},{"t":2000.036,"mix":907},{"t":2001.697,"mix":908},{"t":2003.373,"mix":909},{"t":2004.962,
"mix":910},{"t":2006.549,"mix":911},{"t":2008.245,"mix":912},{"t":2009.712,"mix":913},{"t":2011.405,
"mix":914},{"t":2012.921,"mix":915},{"t":2014.555,"mix":916},{"t":2016.122,"mix":917},{"t":2017.815,
"mix":918},{"t":2019.444,"mix":919},{"t":2021.051,"mix":920},{"t":2022.711,"mix":921},{"t":2024.423,
"mix":922},{"t":2026.196,"mix":923},{"t":2027.862,"mix":924},{"t":2029.726,"mix":925},{"t":2031.406,
"mix":926},{"t":2033.132,"mix":927},{"t":2034.856,"mix":928},{"t":2036.621,"mix":929},{"t":2038.311,
"mix":930},{"t":2040.054,"mix":931},{"t":2041.684,"mix":932},{"t":2043.381,"mix":933},{"t":2045.109,
"mix":934},{"t":2046.767,"mix":935},{"t":2048.407,"mix":936},{"t":2050.085,"mix":937},{"t":2051.686,
"mix":938},{"t":2053.535,"mix":939},{"t":2055.352,"mix":940},{"t":2057.108,"mix":941},{"t":2059.043,
"mix":942},{"t":2060.859,"mix":943},{"t":2062.7,"mix":944},{"t":2064.504,"mix":945},{"t":2066.379,
"mix":946},{"t":2068.163,"mix":947},{"t":2069.793,"mix":948},{"t":2071.368,"mix":949},{"t":2072.931,
"mix":950},{"t":2074.551,"mix":951},{"t":2076.149,"mix":952},{"t":2077.764,"mix":953},{"t":2079.37,
"mix":954},{"t":2081.129,"mix":955},{"t":2082.898,"mix":956},{"t":2084.779,"mix":957},{"t":2086.528,
"mix":958},{"t":2088.203,"mix":959},{"t":2089.864,"mix":960},{"t":2091.637,"mix":961},{"t":2093.366,
"mix":962},{"t":2094.985,"mix":963},{"t":2096.686,"mix":964},{"t":2098.421,"mix":965},{"t":2100.1,
"mix":966},{"t":2101.854,"mix":967},{"t":2103.486,"mix":968},{"t":2105.197,"mix":969},{"t":2106.879,
"mix":970},{"t":2108.664,"mix":971},{"t":2110.374,"mix":972},{"t":2112.083,"mix":973},{"t":2113.801,
"mix":974},{"t":2115.62,"mix":975},{"t":2117.299,"mix":976},{"t":2119.123,"mix":977},{"t":2120.698,
"mix":978},{"t":2122.524,"mix":979},{"t":2124.202,"mix":980},{"t":2126.043,"mix":981},{"t":2127.793,
"mix":982},{"t":2129.473,"mix":983},{"t":2131.392,"mix":984},{"t":2133.199,"mix":985},{"t":2135.128,
"mix":986},{"t":2137.032,"mix":987},{"t":2138.954,"mix":988},{"t":2141.046,"mix":989},{"t":2143.186,
"mix":990},{"t":2145.577,"mix":991},{"t":2148.758,"mix":992},{"t":2151.281,"mix":993},{"t":2153.518,
"mix":994},{"t":2155.592,"mix":995},{"t":2157.804,"mix":996},{"t":2159.981,"mix":997},{"t":2162.132,
"mix":998},{"t":2164.195,"mix":999},{"t":2166.496,"mix":1000},{"t":2168.626,"mix":1001},{"t":2170.711,
"mix":1002},{"t":2172.823,"mix":1003},{"t":2175.051,"mix":1004},{"t":2177.002,"mix":1005},
{"t":2179.217,"mix":1006},{"t":2181.372,"mix":1007},{"t":2183.196,"mix":1008},{"t":2185.371,
"mix":1009},{"t":2187.491,"mix":1010},{"t":2189.63,"mix":1011},{"t":2191.681,"mix":1012},
{"t":2193.678,"mix":1013},{"t":2195.69,"mix":1014},{"t":2197.6,"mix":1015},{"t":2199.437,
"mix":1016},{"t":2201.296,"mix":1017},{"t":2203.051,"mix":1018},{"t":2205.047,"mix":1019},
{"t":2206.763,"mix":1020},{"t":2208.499,"mix":1021},{"t":2210.18,"mix":1022},{"t":2211.897,
"mix":1023},{"t":2213.629,"mix":1024},{"t":2215.292,"mix":1025},{"t":2217.007,"mix":1026},
{"t":2218.722,"mix":1027},{"t":2220.252,"mix":1028},{"t":2221.865,"mix":1029},{"t":2223.673,
"mix":1030},{"t":2225.316,"mix":1031},{"t":2226.958,"mix":1032},{"t":2228.53,"mix":1033},
{"t":2230.301,"mix":1034},{"t":2231.994,"mix":1035},{"t":2233.656,"mix":1036},{"t":2235.312,
"mix":1037},{"t":2236.964,"mix":1038},{"t":2238.583,"mix":1039},{"t":2240.266,"mix":1040},
{"t":2241.916,"mix":1041},{"t":2243.562,"mix":1042},{"t":2245.271,"mix":1043},{"t":2246.878,
"mix":1044},{"t":2248.535,"mix":1045},{"t":2250.116,"mix":1046},{"t":2251.762,"mix":1047},
{"t":2253.414,"mix":1048},{"t":2255.034,"mix":1049},{"t":2256.581,"mix":1050},{"t":2258.321,
"mix":1051},{"t":2260.004,"mix":1052},{"t":2261.712,"mix":1053},{"t":2263.424,"mix":1054},
{"t":2265.028,"mix":1055},{"t":2266.767,"mix":1056},{"t":2268.474,"mix":1057},{"t":2270.198,
"mix":1058},{"t":2271.927,"mix":1059},{"t":2273.557,"mix":1060},{"t":2275.3,"mix":1061},{"t":2277.045,
"mix":1062},{"t":2278.751,"mix":1063},{"t":2280.557,"mix":1064},{"t":2282.34,"mix":1065},
{"t":2284.096,"mix":1066},{"t":2285.807,"mix":1067},{"t":2287.53,"mix":1068},{"t":2289.196,
"mix":1069},{"t":2290.94,"mix":1070},{"t":2292.671,"mix":1071},{"t":2294.354,"mix":1072},
{"t":2296.185,"mix":1073},{"t":2298.37,"mix":1074},{"t":2300.459,"mix":1075},{"t":2302.303,
"mix":1076},{"t":2304.118,"mix":1077},{"t":2305.859,"mix":1078},{"t":2307.613,"mix":1079},
{"t":2309.358,"mix":1080},{"t":2311.092,"mix":1081},{"t":2313.01,"mix":1082},{"t":2314.885,
"mix":1083},{"t":2316.851,"mix":1084},{"t":2318.814,"mix":1085},{"t":2321.088,"mix":1086},
{"t":2323.299,"mix":1087},{"t":2325.535,"mix":1088},{"t":2327.803,"mix":1089},{"t":2330.036,
"mix":1090},{"t":2332.682,"mix":1091},{"t":2336.988,"mix":1092},{"t":2340.31,"mix":1093},
{"t":2343.566,"mix":1094},{"t":2347.285,"mix":1095},{"t":2351.163,"mix":1096},{"t":2354.902,
"mix":1097},{"t":2358.943,"mix":1098},{"t":2363.346,"mix":1099},{"t":2367.48,"mix":1100},
{"t":2371.446,"mix":1101},{"t":2375.329,"mix":1102},{"t":2379.104,"mix":1103},{"t":2383.144,
"mix":1104},{"t":2387.487,"mix":1105},{"t":2392.428,"mix":1106},{"t":2397.362,"mix":1107},
{"t":2399.89,"mix":1108},{"t":2401.987,"mix":1109},{"t":2403.84,"mix":1110},{"t":2405.773,
"mix":1111},{"t":2407.534,"mix":1112},{"t":2409.427,"mix":1113},{"t":2411.226,"mix":1114},
{"t":2412.982,"mix":1115},{"t":2414.856,"mix":1116},{"t":2416.799,"mix":1117},{"t":2418.582,
"mix":1118},{"t":2420.326,"mix":1119},{"t":2421.89,"mix":1120},{"t":2423.756,"mix":1121},
{"t":2425.5,"mix":1122},{"t":2427.554,"mix":1123},{"t":2429.47,"mix":1124},{"t":2431.247,
"mix":1125},{"t":2433.076,"mix":1126},{"t":2435.147,"mix":1127},{"t":2437.128,"mix":1128},
{"t":2439.096,"mix":1129},{"t":2441.033,"mix":1130},{"t":2442.958,"mix":1131},{"t":2444.773,
"mix":1132},{"t":2446.459,"mix":1133},{"t":2448.133,"mix":1134},{"t":2449.86,"mix":1135},
{"t":2451.423,"mix":1136},{"t":2453.015,"mix":1137},{"t":2454.713,"mix":1138},{"t":2456.56,
"mix":1139},{"t":2458.372,"mix":1140},{"t":2460.229,"mix":1141},{"t":2461.892,"mix":1142},
{"t":2463.628,"mix":1143},{"t":2465.359,"mix":1144},{"t":2467.067,"mix":1145},{"t":2468.933,
"mix":1146},{"t":2470.563,"mix":1147},{"t":2472.227,"mix":1148},{"t":2474.075,"mix":1149},
{"t":2475.707,"mix":1150},{"t":2477.34,"mix":1151},{"t":2479.069,"mix":1152},{"t":2480.701,
"mix":1153},{"t":2482.362,"mix":1154},{"t":2484.072,"mix":1155},{"t":2485.777,"mix":1156},
{"t":2487.419,"mix":1157},{"t":2489.223,"mix":1158},{"t":2490.884,"mix":1159},{"t":2492.578,
"mix":1160},{"t":2494.409,"mix":1161},{"t":2495.969,"mix":1162},{"t":2497.757,"mix":1163},
{"t":2499.399,"mix":1164},{"t":2501.221,"mix":1165},{"t":2503.001,"mix":1166},{"t":2504.642,
"mix":1167},{"t":2506.467,"mix":1168},{"t":2508.381,"mix":1169},{"t":2510.212,"mix":1170},
{"t":2512.064,"mix":1171},{"t":2514.009,"mix":1172},{"t":2516.314,"mix":1173},{"t":2518.403,
"mix":1174},{"t":2520.469,"mix":1175},{"t":2522.458,"mix":1176},{"t":2524.647,"mix":1177},
{"t":2526.589,"mix":1178},{"t":2528.545,"mix":1179},{"t":2530.673,"mix":1180},{"t":2532.743,
"mix":1181},{"t":2534.84,"mix":1182},{"t":2536.896,"mix":1183},{"t":2538.911,"mix":1184},
{"t":2540.695,"mix":1185},{"t":2542.644,"mix":1186},{"t":2544.415,"mix":1187},{"t":2546.019,
"mix":1188},{"t":2547.588,"mix":1189},{"t":2549.101,"mix":1190},{"t":2550.5,"mix":1191},{"t":2551.696,
"mix":1192},{"t":2552.916,"mix":1193},{"t":2553.995,"mix":1194},{"t":2555.334,"mix":1195},
{"t":2556.404,"mix":1196},{"t":2557.523,"mix":1197},{"t":2558.68,"mix":1198},{"t":2559.836,
"mix":1199},{"t":2560.848,"mix":1200},{"t":2561.936,"mix":1201},{"t":2562.945,"mix":1202},
{"t":2564.067,"mix":1203},{"t":2565.058,"mix":1204},{"t":2566.121,"mix":1205},{"t":2567.17,
"mix":1206},{"t":2568.219,"mix":1207},{"t":2569.22,"mix":1208},{"t":2570.259,"mix":1209},
{"t":2571.295,"mix":1210},{"t":2572.425,"mix":1211},{"t":2573.563,"mix":1212},{"t":2575.172,
"mix":1213},{"t":2576.868,"mix":1214},{"t":2578.534,"mix":1215},{"t":2580.101,"mix":1216},
{"t":2581.541,"mix":1217},{"t":2583.257,"mix":1218},{"t":2585.192,"mix":1219},{"t":2586.411,
"mix":1220},{"t":2587.604,"mix":1221},{"t":2588.749,"mix":1222},{"t":2589.651,"mix":1223},
{"t":2590.666,"mix":1224},{"t":2591.587,"mix":1225},{"t":2592.566,"mix":1226},{"t":2593.565,
"mix":1227},{"t":2594.528,"mix":1228},{"t":2595.476,"mix":1229},{"t":2596.416,"mix":1230},
{"t":2597.334,"mix":1231},{"t":2598.303,"mix":1232},{"t":2599.383,"mix":1233},{"t":2600.374,
"mix":1234},{"t":2601.372,"mix":1235},{"t":2602.387,"mix":1236},{"t":2603.378,"mix":1237},
{"t":2604.51,"mix":1238},{"t":2605.567,"mix":1239},{"t":2606.573,"mix":1240},{"t":2607.724,
"mix":1241},{"t":2608.782,"mix":1242},{"t":2609.814,"mix":1243},{"t":2610.803,"mix":1244},
{"t":2611.822,"mix":1245},{"t":2612.953,"mix":1246},{"t":2614.009,"mix":1247},{"t":2614.973,
"mix":1248},{"t":2615.989,"mix":1249},{"t":2617.01,"mix":1250},{"t":2618.195,"mix":1251},
{"t":2619.293,"mix":1252},{"t":2620.332,"mix":1253},{"t":2621.359,"mix":1254},{"t":2622.46,
"mix":1255},{"t":2623.435,"mix":1256},{"t":2624.58,"mix":1257},{"t":2625.49,"mix":1258},{"t":2626.664,
"mix":1259},{"t":2627.896,"mix":1260},{"t":2629.184,"mix":1261},{"t":2630.506,"mix":1262},
{"t":2632.439,"mix":1263},{"t":2652.701,"mix":1264}]
'''  # Use the full JSON
timestamps1 = json.loads(timestamps_json)

def parse_start_time_from_url(youtube_url):
    parsed_url = urlparse.urlparse(youtube_url)
    query_params = urlparse.parse_qs(parsed_url.query)
    start_time = query_params.get('t', ['0'])[0]  # Default to '0' if not provided
    if 's' in start_time:
        # Extract time in seconds from the URL parameter
        time_seconds = int(re.search(r'\d+', start_time).group())
        return time_seconds
    else:
        return int(start_time)

def download_audio_with_retries(youtube_url, output_path, max_retries=5):
    retry_count = 0
    while retry_count < max_retries:
        try:
            yt = YouTube(youtube_url)
            audio_stream = yt.streams.filter(only_audio=True).first()
            if not audio_stream:
                print("No audio stream found.")
                return None
            # Download the audio stream directly without conversion
            downloaded_file = audio_stream.download(filename=output_path)
            return downloaded_file
        except Exception as e:
            print(f"Attempt {retry_count + 1} failed: {str(e)}")
            time.sleep(5)  # wait 5 seconds before retrying
            retry_count += 1
    print("Failed to download after several retries.")
    return None

def convert_time_to_seconds(time):
    if isinstance(time, str) and ':' in time:
        minutes, seconds = map(int, time.split(':'))
        return minutes * 60 + seconds
    elif isinstance(time, (int, float)):
        return time
    else:
        raise ValueError("Time format must be a string 'MM:SS' or a number representing seconds")

def convert_to_ogg(input_path, output_path, start_time=0, end_time=None):
    try:
        # Convert start time to seconds and add offset
        start_total_seconds = convert_time_to_seconds(start_time) + 0.15
        
        # Initialize the ffmpeg command
        command = [
            'ffmpeg', '-ss', str(start_total_seconds), '-i', input_path,
            '-c:a', 'libvorbis', '-q:a', '5'
        ]
        
        if end_time is not None:
            # Convert end time to seconds
            end_total_seconds = convert_time_to_seconds(end_time)
            # Calculate the duration of the clip
            clip_duration = end_total_seconds - start_total_seconds
            command.extend(['-t', str(clip_duration)])
        
        command.append(output_path)
        
        subprocess.run(command, check=True)
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")
        return None
    except ValueError as e:
        print(f"Invalid time format: {e}")
        return None

# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=Lu3T0f-H3JI'
youtube_url2 = 'https://www.youtube.com/watch?v=UMAROdS_ckQ'

start_time1 = 46.46
start_time2 = parse_start_time_from_url(youtube_url2)
end_time1 = "44:45"
end_time2 = None

# Download audio files
audio_path1 = download_audio_with_retries(youtube_url1, 'youtube_audio1.mp4')
audio_path2 = download_audio_with_retries(youtube_url2, 'youtube_audio2.mp4')


# Convert to OGG
ogg_path1 = "youtube_audio1.ogg"
ogg_path2 = "youtube_audio2.ogg"
convert_to_ogg(audio_path1, ogg_path1, start_time1, end_time1)
convert_to_ogg(audio_path2, ogg_path2, start_time2, end_time2)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

'youtube_audio2.ogg'

In [10]:
import gc

HOP_LENGTH = 512
HOP_LENGTH_1 = 1024
HOP_LENGTH_2 = 256
OVERLAP_HOP = 256

def load_and_preprocess_audio_ORIGINAL(audio_path, target_sr=11025):
    # Load audio file at a reduced sample rate
    y, sr = librosa.load(audio_path, sr=target_sr)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256)
    log_S = librosa.power_to_db(S, ref=np.max)
    return log_S.T, sr

def load_and_preprocess_audio(audio_path, target_sr=11025, hop_length=HOP_LENGTH):
    # Load audio file at a higher sample rate for better temporal resolution
    y, sr = librosa.load(audio_path, sr=target_sr)
    
    # Harmonic-Percussive Source Separation (HPSS) --- cutting for resource savings
    #y_harmonic, y_percussive = librosa.effects.hpss(y)
    
    # Mel-spectrogram for harmonic component
    #S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=64, hop_length=HOP_LENGTH)
    #log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
    
    # Constant-Q Transform for better frequency resolution
    #CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    # Combine features
    #combined_features = np.vstack((log_S_harmonic, CQT))
    #print(f"Combined features shape: {combined_features.shape}")
    
    #return combined_features.T, sr

    #BELOW IS STRIPPED VERSION OF ABOVE
    y, sr = librosa.load(audio_path, sr=sr)
    
    # Mel-spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=512, hop_length=HOP_LENGTH)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Constant-Q Transform
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    combined_features = np.vstack((log_S, CQT))
    return combined_features.T, sr


def load_and_preprocess_audio_INCREMENTAL(audio_path, target_sr=11025, chunk_duration=10):
    # Load the audio file in chunks
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Harmonic-Percussive Source Separation (HPSS)
        y_harmonic, y_percussive = librosa.effects.hpss(y_chunk)
        
        # Mel-spectrogram for harmonic component
        S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=256)
        log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
        
        # Constant-Q Transform for better frequency resolution
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr)), ref=np.max)
        
        # Combine features
        combined_chunk_features = np.vstack((log_S_harmonic, CQT))
        
        combined_features.append(combined_chunk_features.T)
        
        # Print progress
        if (i + 1) % progress_step == 0:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    
    return combined_features, sr

def load_and_preprocess_audio_OVERLAP(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows
        chunk_features = []
        for offset in range(0, HOP_LENGTH, OVERLAP_HOP):
            if offset > 0:
                y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
            else:
                y_shifted = y_chunk
            
            # HPSS
            y_harmonic, y_percussive = librosa.effects.hpss(y_shifted)
            
            # Mel-spectrogram
            S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
            log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
            
            # Constant-Q Transform
            CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_shifted, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
            
            # Combine features for this offset
            combined_chunk_features = np.vstack((log_S_harmonic, CQT))
            chunk_features.append(combined_chunk_features.T)
            del y_shifted, y_harmonic, y_percussive, S_harmonic, log_S_harmonic, CQT
            gc.collect()

        combined_chunk_features = np.concatenate(chunk_features, axis=1)
        combined_features.append(combined_chunk_features)
        del chunk_features
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

#low and high averageed hoplengths
def load_and_preprocess_audio_AVERAGE(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH_2 // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows for both hop lengths
        chunk_features_list = []
        max_length = 0
        for hop_length in [HOP_LENGTH_1, HOP_LENGTH_2]:
            chunk_features = []
            for offset in range(0, hop_length, OVERLAP_HOP):
                if offset > 0:
                    y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
                else:
                    y_shifted = y_chunk
                
                # Mel-spectrogram
                S = librosa.feature.melspectrogram(y=y_shifted, sr=sr, n_mels=n_mels, hop_length=hop_length)
                log_S = librosa.power_to_db(S, ref=np.max)
                
                chunk_features.append(log_S.T)
                del y_shifted, S, log_S
                gc.collect()

            # Find the maximum length of the feature matrices
            max_length = max(max_length, max(f.shape[0] for f in chunk_features))
            chunk_features_list.append(chunk_features)
            del chunk_features
            gc.collect()
        
        # Resize all feature matrices to the maximum length and average
        resized_chunk_features_list = []
        for features in chunk_features_list:
            resized_features = [resize(f, (max_length, f.shape[1]), anti_aliasing=True) for f in features]
            averaged_chunk_features = np.mean(resized_features, axis=0)
            resized_chunk_features_list.append(averaged_chunk_features)
        
        # Average the features from different hop lengths
        combined_chunk_features = np.mean(resized_chunk_features_list, axis=0)
        combined_features.append(combined_chunk_features)
        del chunk_features_list, resized_chunk_features_list
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

def load_and_preprocess_audio_INCREMENTAL_NOHPSS(audio_path, target_sr=11025, chunk_duration=10, n_mels=512, mel_weight=1.0, cqt_weight=1.0, tempo_weight=1.0):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Mel-spectrogram
        S = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
        log_S = librosa.power_to_db(S, ref=np.max)
        
        # Constant-Q Transform
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

        # Tempogram
        onset_env = librosa.onset.onset_strength(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
        tempogram = librosa.feature.tempogram(onset_envelope=onset_env, sr=sr, hop_length=HOP_LENGTH)

        # Ensure the same length for both arrays
        min_length = min(log_S.shape[1], CQT.shape[1])
        log_S = log_S[:, :min_length]
        CQT = CQT[:, :min_length]
        
        # Normalize features
        log_S = normalize_features(log_S)
        CQT = normalize_features(CQT)

        log_S *= mel_weight
        CQT *= cqt_weight
        tempogram *= tempo_weight
        
        # Combine features
        combined_chunk_features = np.vstack((log_S, CQT))

        combined_features.append(combined_chunk_features.T)
        del log_S, CQT, y_chunk, S, combined_chunk_features
        gc.collect()
        
        # Print progress
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr
    
def normalize_features(features, epsilon=1e-8):
    mean = np.mean(features, axis=0)
    std_dev = np.std(features, axis=0)
    return (features - mean) / (std_dev + epsilon)

    
# Load and preprocess both audio recordings in OGG format
S1, sr1 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path1, mel_weight=1.0, cqt_weight=1.0, tempo_weight=2.0)

Total chunks: 264, Progress step: 26
Processed 10% of chunks
Processed 20% of chunks
Processed 30% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 89% of chunks
Processed 98% of chunks
Processed 100% of chunks
Combined features shape: (56989, 596)


In [11]:
#making own step so no need to re-run audio1 for multiple recordings.
S2, sr2 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path2, mel_weight=1.0, cqt_weight=1.0, tempo_weight=2.0)

Total chunks: 274, Progress step: 27
Processed 10% of chunks
Processed 20% of chunks
Processed 30% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 89% of chunks
Processed 99% of chunks
Processed 100% of chunks
Combined features shape: (59034, 596)


In [14]:
from fastdtw import fastdtw

def dynamic_time_warping_approx(S1, S2):
    distance, path = fastdtw(S1, S2)
    return path

warping_path = dynamic_time_warping_approx(S1, S2)

In [24]:
def adjust_timestamps(wp, timestamps, sr):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    
    for entry in timestamps:
        original_frame = int((entry['t']) * sr / HOP_LENGTH)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * HOP_LENGTH / sr
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    
    # Ensure the last timestamp is included and set "t" to 9999
    if timestamps:
        last_entry = timestamps[-1]
        last_entry_adjusted = {"t": 9999, "mix": last_entry['mix']}
        if adjusted_timestamps and adjusted_timestamps[-1]['mix'] == last_entry['mix']:
            adjusted_timestamps[-1] = last_entry_adjusted
        else:
            adjusted_timestamps.append(last_entry_adjusted)
    
    return adjusted_timestamps

# Sample JSON timestamps for the first recording (assumed already loaded)
adjusted_timestamps = adjust_timestamps(warping_path, timestamps1, sr1)

In [25]:
def calculate_ratios(timestamps):
    ratios = []
    for i in range(1, len(timestamps)):
        current_ratio = abs(timestamps[i]['t'] - timestamps[i-1]['t']) if timestamps[i-1]['t'] != 0 else 0
        ratios.append(current_ratio)
    return ratios

def compare_and_flag_changes(adjusted_timestamps, original_timestamps, audio_length, neighbor_count=5):
    # Set last timestamp as per new requirement
    adjusted_timestamps[-1]['t'] = audio_length + 1

    # Calculate differences and ratios
    adjusted_ratios = calculate_ratios(adjusted_timestamps)
    original_ratios = calculate_ratios(original_timestamps)

    # Array to hold timestamps that are significantly different
    flagged_timestamps = []

    # Analyze ratios for significant changes
    for i in range(len(adjusted_ratios) - 1):  # Ignore the last timestamp in comparison
        start = max(0, i - neighbor_count)
        end = min(len(original_ratios) - 1, i + neighbor_count + 1)  # Avoid including the last in comparison
        
        # Calculate neighborhood average without including out-of-range values
        neighborhood_original = original_ratios[start:end]
        if not neighborhood_original:
            continue
        neighborhood_average = np.mean(neighborhood_original)
        
        # Check if the current adjusted ratio is significantly different
        if adjusted_ratios[i] > 1.5 * neighborhood_average:
            flagged_timestamps.append({
                "mix": adjusted_timestamps[i]['mix'],
                "original_ratio": original_ratios[i] if i < len(original_ratios) else 0,
                "adjusted_ratio": adjusted_ratios[i],
                "average_neighbors": neighborhood_average
            })

    return flagged_timestamps

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']

# Initialize an empty list to store the new adjusted timestamps
new_adjusted_timestamps = []
previous_t = None  # Variable to hold the previous timestamp

for item in adjusted_timestamps:
    adjusted_t = round(item["t"] + initial_offset, 3)
    new_adjusted_timestamps.append({"t": adjusted_t, "mix": item["mix"]})
    
    # Check if the previous timestamp is defined and compare the current timestamp with the previous one
    if previous_t is not None and (adjusted_t - previous_t < 0.5):
        difference = adjusted_t - previous_t
        print(f"Close timestamps found: Mix: {item['mix']}, Difference: {difference:.3f}, Previous - {previous_t}, Current - {adjusted_t}")
    
    # Update the previous_t to the current timestamp for the next iteration
    previous_t = adjusted_t


# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps, indent=4))
# Here we adjust to show the total offset from the original video start
full_offset = abs(initial_offset) + abs(start_time2)
print(f"Total Offset from Video Start: {full_offset}, initial {initial_offset} + start_time2 {start_time2}")

#Flag anything that exceeds 50% difference compared to neighboring measures.
audio_length = librosa.get_duration(path=ogg_path2)
flagged_timestamps = compare_and_flag_changes(new_adjusted_timestamps, timestamps1, audio_length)
print("Flagged Timestamps:")
for ft in flagged_timestamps:
    print(f"Mix: {ft['mix']}, Original Ratio: {ft['original_ratio']:.3f}, Adjusted Ratio: {ft['adjusted_ratio']:.3f}, Neighbors' Avg.: {ft['average_neighbors']:.3f}")

# Cleanup downloaded and converted files
#os.remove(audio_path1)
#os.remove(audio_path2)
#os.remove(ogg_path1)
os.remove(ogg_path2)

[
    {
        "t": 0.0,
        "mix": 0
    },
    {
        "t": 7.663,
        "mix": 1
    },
    {
        "t": 12.725,
        "mix": 2
    },
    {
        "t": 17.554,
        "mix": 3
    },
    {
        "t": 22.895,
        "mix": 4
    },
    {
        "t": 27.632,
        "mix": 5
    },
    {
        "t": 32.461,
        "mix": 6
    },
    {
        "t": 37.013,
        "mix": 7
    },
    {
        "t": 45.047,
        "mix": 8
    },
    {
        "t": 49.876,
        "mix": 9
    },
    {
        "t": 54.613,
        "mix": 10
    },
    {
        "t": 59.722,
        "mix": 11
    },
    {
        "t": 65.062,
        "mix": 12
    },
    {
        "t": 69.66,
        "mix": 13
    },
    {
        "t": 74.397,
        "mix": 14
    },
    {
        "t": 79.97,
        "mix": 15
    },
    {
        "t": 85.124,
        "mix": 16
    },
    {
        "t": 90.093,
        "mix": 17
    },
    {
        "t": 95.248,
        "mix": 18
    },
    {
        "t": 99.892,